<a href="https://colab.research.google.com/github/despoileralert/DSA2101-Project/blob/main/DT_Ensembles.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import os

In [2]:
# drive mount
from google.colab import drive
drive.mount('/content/drive')

shared_folder_path = '/content/drive/MyDrive/time-series-human'

# Example: List the files inside the shared folder
import os
print(os.listdir(shared_folder_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['features.txt', 'activity_labels.txt', 'README.txt', 'sampler.py', 'features_info.txt', 'view.ipynb', 'RawData', 'Train', 'Model Training', 'Test', 'feature_extractor.py']


In [3]:
df = pd.read_csv('/content/drive/MyDrive/time-series-human/Train/X_train.txt', sep=r'\s+', header=None)

new_cols = []
with open('/content/drive/MyDrive/time-series-human/features.txt', 'r') as f:
    for line in f:
        new_cols.append(line.strip())
df.columns = new_cols

In [4]:
df

,tBodyAcc-Mean-1,tBodyAcc-Mean-2,tBodyAcc-Mean-3,tBodyAcc-STD-1,tBodyAcc-STD-2,tBodyAcc-STD-3,tBodyAcc-Mad-1,tBodyAcc-Mad-2,tBodyAcc-Mad-3,tBodyAcc-Max-1,...,fBodyGyroJerkMag-MeanFreq-1,fBodyGyroJerkMag-Skewness-1,fBodyGyroJerkMag-Kurtosis-1,tBodyAcc-AngleWRTGravity-1,tBodyAccJerk-AngleWRTGravity-1,tBodyGyro-AngleWRTGravity-1,tBodyGyroJerk-AngleWRTGravity-1,tXAxisAcc-AngleWRTGravity-1,tYAxisAcc-AngleWRTGravity-1,tZAxisAcc-AngleWRTGravity-1
0,0.043580,-0.005970,-0.035054,-0.995381,-0.988366,-0.937382,-0.995007,-0.988816,-0.953325,-0.794796,...,-0.012236,-0.314848,-0.713308,-0.112754,0.030400,-0.464761,-0.018446,-0.841559,0.179913,-0.051718
1,0.039480,-0.002131,-0.029067,-0.998348,-0.982945,-0.971273,-0.998702,-0.983315,-0.974000,-0.802537,...,0.202804,-0.603199,-0.860677,0.053477,-0.007435,-0.732626,0.703511,-0.845092,0.180261,-0.047436
2,0.039978,-0.005153,-0.022651,-0.995482,-0.977314,-0.984760,-0.996415,-0.975835,-0.985973,-0.798477,...,0.440079,-0.404427,-0.761847,-0.118559,0.177899,0.100699,0.808529,-0.849230,0.180610,-0.042271
3,0.039785,-0.011809,-0.028916,-0.996194,-0.988569,-0.993256,-0.996994,-0.988526,-0.993135,-0.798477,...,0.430891,-0.138373,-0.491604,-0.036788,-0.012892,0.640011,-0.485366,-0.848947,0.181907,-0.040826
4,0.038758,-0.002289,-0.023863,-0.998241,-0.986774,-0.993115,-0.998216,-0.986479,-0.993825,-0.801982,...,0.137735,-0.366214,-0.702490,0.123320,0.122542,0.693578,-0.615971,-0.848164,0.185124,-0.037080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7762,0.048048,-0.042445,-0.065884,-0.195448,-0.278326,-0.219954,-0.282233,-0.305861,-0.357803,0.267874,...,-0.008381,-0.596760,-0.879026,-0.190437,0.829718,0.206972,-0.425619,-0.792292,0.238580,0.056020
7763,0.037639,0.006430,-0.044345,-0.235372,-0.302680,-0.232843,-0.322483,-0.354464,-0.345592,0.181271,...,0.209452,-0.404418,-0.684496,0.064907,0.875679,-0.879033,0.400219,-0.772288,0.252653,0.056252
7764,0.037451,-0.002724,0.021009,-0.218281,-0.378082,-0.076950,-0.304446,-0.400661,-0.193071,0.113141,...,0.237003,0.000207,-0.317314,0.052806,-0.266724,0.864404,0.701169,-0.779566,0.249121,0.047071
7765,0.044011,-0.004536,-0.051242,-0.219202,-0.383350,-0.081035,-0.310419,-0.380233,-0.201007,0.166671,...,0.069366,0.037919,-0.356579,-0.101360,0.700740,0.936674,-0.589479,-0.785603,0.246409,0.031700


In [5]:
# The train test split is done by subject
train_subjects = np.loadtxt("/content/drive/MyDrive/time-series-human/Train/subject_id_train.txt", dtype=int)

test_subjects = np.loadtxt("/content/drive/MyDrive/time-series-human/Test/subject_id_test.txt", dtype=int)

print("Train subjects:", np.unique(train_subjects))
print("Test subjects: ", np.unique(test_subjects))

Train subjects: [ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
Test subjects:  [ 2  4  9 10 12 13 18 20 24]


# Cross Validation Step

In [6]:
import time
import warnings

from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.decomposition import PCA, FactorAnalysis, FastICA, KernelPCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_selection import (
    RFE,
    SelectFromModel,
    SelectKBest,
    VarianceThreshold,
    f_classif,
    mutual_info_classif,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.random_projection import GaussianRandomProjection
from sklearn.tree import DecisionTreeClassifier

In [7]:
#Firstly we handle the train test split

base = "/content/drive/MyDrive/time-series-human"

import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

X = pd.read_csv(os.path.join(base, "Train", "features_train.csv")) #Train set
y = pd.read_csv(os.path.join(base, "Test", "features_test.csv"))   #Test set

X_train, X_test = X.iloc[:, 3:], y.iloc[:, 3:]
y_train, y_test = (X.iloc[:, 2]).to_numpy(), y.iloc[:, 2].to_numpy()
Xgroups, ygroups = (X.iloc[:, 0]).to_numpy(), y.iloc[:, 0].to_numpy()

print(np.unique(Xgroups))
print(np.unique(ygroups))

#Initialize CV method
sgkf = StratifiedGroupKFold(n_splits=5, shuffle = True, random_state = 42)
sgkf.get_n_splits()

groups = np.append(Xgroups, ygroups)
"""print("Xtrain is ", X_train)
print("y_train is ", y_train)"""
for i, (train_index, val_index) in enumerate(sgkf.split(X_train, y_train, Xgroups)):
    print(f"Fold {i}:")
    print(f"  Train: index length = { len(train_index)}")
    print(f"         group={np.unique(groups[train_index])}")
    print(f"  Validation:  index length = {len(val_index)}")
    print(f"         group={np.unique(groups[val_index])}")

[ 1  3  5  6  7  8 11 14 15 16 17 19 21 22 23 25 26 27 28 29 30]
[ 2  4  9 10 12 13 18 20 24]
Fold 0:
  Train: index length = 6239
         group=[ 1  3  5  6  8 11 14 15 16 17 21 22 23 26 27 29 30]
  Validation:  index length = 1528
         group=[ 7 19 25 28]
Fold 1:
  Train: index length = 6184
         group=[ 1  3  5  6  7  8 11 14 15 19 21 22 23 25 26 28 29]
  Validation:  index length = 1583
         group=[16 17 27 30]
Fold 2:
  Train: index length = 6024
         group=[ 3  5  6  7 14 16 17 19 21 22 25 26 27 28 29 30]
  Validation:  index length = 1743
         group=[ 1  8 11 15 23]
Fold 3:
  Train: index length = 6363
         group=[ 1  5  6  7  8 11 15 16 17 19 21 23 25 26 27 28 30]
  Validation:  index length = 1404
         group=[ 3 14 22 29]
Fold 4:
  Train: index length = 6258
         group=[ 1  3  7  8 11 14 15 16 17 19 22 23 25 27 28 29 30]
  Validation:  index length = 1509
         group=[ 5  6 21 26]


In [ ]:
# We measure the performance of base DT and Random Foreset Models.

X_orig = X_train.loc[:, ~X_train.columns.str.startswith("wavelet")]

models = {
    "Decision tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            n_jobs=-1, random_state=42),
}

featuresets = {"X_orig": X_orig, "X_train": X_train}
results = []

# The Matthews correlation coefficient is used in machine learning as a measure of the quality of binary and multiclass classifications.
# It takes into account true and false positives and negatives and is generally regarded as a balanced measure which can be used even if the classes are of very different sizes.
# The MCC is in essence a correlation coefficient value between -1 and +1.
# A coefficient of +1 represents a perfect prediction, 0 an average random prediction and -1 an inverse prediction.
# The statistic is also known as the phi coefficient. [source: Wikipedia]

scoringmetrics = ["balanced_accuracy", "f1_macro", "matthews_corrcoef"]
#Train on original features + extended features
for feature in featuresets:
  for estname, estimator in models.items():
    s = cross_validate(estimator, featuresets[feature], y_train,
                      groups = Xgroups,
                      scoring = scoringmetrics,
                      cv = sgkf,
                      verbose = 5)
    entry = {"featureset": feature, "model": estname}
    for i in s:
      entry[i] = np.median(s[i])
    entry["model"] = estname
    results.append(entry)
  print(results)

finaltable = pd.DataFrame(results).round(3)
finaltable

[CV] END  balanced_accuracy: (test=0.688) f1_macro: (test=0.682) matthews_corrcoef: (test=0.769) total time=  10.3s
[CV] END  balanced_accuracy: (test=0.746) f1_macro: (test=0.732) matthews_corrcoef: (test=0.805) total time=   6.6s
[CV] END  balanced_accuracy: (test=0.744) f1_macro: (test=0.741) matthews_corrcoef: (test=0.853) total time=   6.1s
[CV] END  balanced_accuracy: (test=0.648) f1_macro: (test=0.652) matthews_corrcoef: (test=0.784) total time=   8.6s
[CV] END  balanced_accuracy: (test=0.620) f1_macro: (test=0.604) matthews_corrcoef: (test=0.793) total time=   9.8s


[Parallel(n_jobs=1)]: Done   5 out of   5 | elapsed:   41.5s finished


[CV] END  balanced_accuracy: (test=0.764) f1_macro: (test=0.743) matthews_corrcoef: (test=0.853) total time=  40.5s
[CV] END  balanced_accuracy: (test=0.827) f1_macro: (test=0.833) matthews_corrcoef: (test=0.909) total time=  41.4s
[CV] END  balanced_accuracy: (test=0.822) f1_macro: (test=0.830) matthews_corrcoef: (test=0.950) total time=  38.1s


# Correlation Filter Step

In [ ]:
#Now we make a correlation filter
main = pd.concat([X, y], axis=0)
correlation_matrix = main.corr()
print(correlation_matrix)

# Assume 'correlation_matrix' is calculated as above
# Assume 'threshold' is defined (e.g., threshold = 0.9)
threshold = 0.9
# Use absolute correlation
abs_corr_matrix = correlation_matrix.abs()

# Get the upper triangle of the correlation matrix (excluding the diagonal)
upper_triangle = abs_corr_matrix.where(np.triu(np.ones(abs_corr_matrix.shape), k=1).astype(bool))

# Find features with correlation above the threshold
to_drop = set() # Use a set to avoid duplicates
for column in upper_triangle.columns:
    highly_correlated_with = upper_triangle.index[upper_triangle[column] > threshold].tolist()
    if highly_correlated_with:
        # Basic strategy: Drop the current column if it's highly correlated with any previous one
        # More sophisticated strategies could be implemented here
        # For simplicity, let's collect the column name itself if it correlates highly
        # with any feature already checked (index < column)
        if any(upper_triangle[column] > threshold):
             to_drop.add(column) # Example: Add the second feature in the pair

print(f"Features to drop (based on threshold {threshold}): {list(to_drop)}")

# Drop the selected features from the original DataFrame
df_reduced = df.drop(columns=list(to_drop))
print(f"Original number of features: {df.shape[1]}")
print(f"Number of features after correlation filtering: {df_reduced.shape[1]}")

# Code: https://apxml.com/courses/intro-feature-engineering/chapter-6-feature-selection/filter-correlation-analysis